<a href="https://colab.research.google.com/github/1Poras1/INDIA-SPACE-LAB-ISL-2026/blob/Guidance-and-Path-Tracking-Using-anAutonomous-Boat/ISL_Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**# Task 2: Understanding Guidance & Path Tracking Using an Autonomous Boat**


# imp required libraries :-

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon
from IPython.display import HTML, display
from ipywidgets import interact, FloatSlider

#Creating boat shape :-

In [3]:
# 🚤 Create Boat Shape

def create_boat(x, y, theta, scale=0.8):

    # Triangle boat shape
    boat = np.array([
        [1.5, 0],
        [-1, 0.7],
        [-0.5, 0],
        [-1, -0.7]
    ]) * scale

    # Rotation matrix
    R = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    rotated_boat = boat @ R.T

    rotated_boat[:,0] += x
    rotated_boat[:,1] += y

    return rotated_boat

##Activating the guidance algorithm for the boat :-

In [4]:
# 🚤 Interactive Boat Guidance
def boat_guidance(kp=1.0,
                  kd=0.0,
                  current_x=0.3,
                  current_y=0.3):

    dt = 0.2
    T = 40

    t = np.arange(0, T, dt)

    # Desired path
    path_x = t
    path_y = 8*np.sin(0.2*t)

    # Initial conditions
    x, y = -5, -10
    vx, vy = 0, 0

    traj_x = []
    traj_y = []
    headings = []


    # SIMULATION LOOP
    for i in range(len(t)):

        # Tracking error
        dx = path_x[i] - x
        dy = path_y[i] - y

        # Guidance law
        ax = kp*dx - kd*vx + current_x
        ay = kp*dy - kd*vy + current_y

        # Velocity update
        vx += ax*dt
        vy += ay*dt

        # Position update
        x += vx*dt
        y += vy*dt

        traj_x.append(x)
        traj_y.append(y)

        headings.append(np.arctan2(vy, vx))



    # PLOT
    plt.figure(figsize=(10,6))

    plt.plot(path_x,
             path_y,
             '--',
             linewidth=2,
             label='Desired Path')

    plt.plot(traj_x,
             traj_y,
             linewidth=3,
             label='Boat Trajectory')

    # Start and End
    plt.scatter(traj_x[0], traj_y[0], s=100, label='Start')
    plt.scatter(traj_x[-1], traj_y[-1], s=100, label='End')


    # DRAW BOAT
    boat_shape = create_boat(
        traj_x[-1],
        traj_y[-1],
        headings[-1]
    )

    boat_patch = Polygon(boat_shape,
                         closed=True)

    plt.gca().add_patch(boat_patch)


    # WIND ARROW
    plt.quiver(traj_x[-1],
               traj_y[-1],
               current_x,
               current_y,
               scale=5)

    plt.title(
        f'Boat Guidance | Gain={kp} | Damping={kd}'
    )

    plt.xlabel('X Position')
    plt.ylabel('Y Position')

    plt.grid(True)
    plt.legend()

    plt.xlim(min(path_x)-5, max(path_x)+5)
    plt.ylim(min(path_y)-15, max(path_y)+15)

    plt.show()

#Slider to tune guidance algorithm for the boat :-

In [5]:
interact(
    boat_guidance,

    kp=FloatSlider(
        value=1.0,
        min=0.1,
        max=2.0,
        step=0.1,
        description='Gain'
    ),

    kd=FloatSlider(
        value=0.0,
        min=0.0,
        max=3.0,
        step=0.1,
        description='Damping'
    ),

    current_x=FloatSlider(
        value=0.3,
        min=-1.0,
        max=1.0,
        step=0.1,
        description='Current X'
    ),

    current_y=FloatSlider(
        value=0.3,
        min=-1.0,
        max=1.0,
        step=0.1,
        description='Current Y'
    )
)

interactive(children=(FloatSlider(value=1.0, description='Gain', max=2.0, min=0.1), FloatSlider(value=0.0, des…

<function __main__.boat_guidance(kp=1.0, kd=0.0, current_x=0.3, current_y=0.3)>

#Animation of the boat :-

In [6]:
# 🚤 ANIMATED BOAT GUIDANCE :-
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def animate_boat(kp=1.0,
                 kd=0.0,
                 current_x=0.3,
                 current_y=0.3):

    dt = 0.2
    T = 35

    t = np.arange(0, T, dt)

    # Desired path
    path_x = t
    path_y = 8*np.sin(0.2*t)

    # Initial state
    x, y = -5, -10
    vx, vy = 0, 0

    traj_x = []
    traj_y = []
    headings = []


    # SIMULATION :-

    for i in range(len(t)):

        dx = path_x[i] - x
        dy = path_y[i] - y

        # Guidance law
        ax = kp*dx - kd*vx + current_x
        ay = kp*dy - kd*vy + current_y

        # Velocity update
        vx += ax*dt
        vy += ay*dt

        # Position update
        x += vx*dt
        y += vy*dt

        traj_x.append(x)
        traj_y.append(y)

        headings.append(np.arctan2(vy, vx))


    # CREATE FIGURE :-
    fig, ax = plt.subplots(figsize=(10,6))

    ax.set_xlim(min(path_x)-5, max(path_x)+5)
    ax.set_ylim(min(path_y)-15, max(path_y)+15)

    ax.set_title('Animated Boat Guidance')
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')

    ax.grid(True)

    # Desired path
    ax.plot(path_x,
            path_y,
            '--',
            linewidth=2,
            label='Desired Path')

    # Boat trajectory line
    traj_line, = ax.plot([], [], linewidth=3,
                         label='Boat Trajectory')

    # Initial boat
    initial_boat = create_boat(
        traj_x[0],
        traj_y[0],
        headings[0]
    )

    boat_patch = Polygon(initial_boat,
                         closed=True)

    ax.add_patch(boat_patch)


    # WATER CURRENT ARROWS :-
    X, Y = np.meshgrid(
        np.linspace(min(path_x)-5, max(path_x)+5, 15),
        np.linspace(min(path_y)-15, max(path_y)+15, 10)
    )

    U = current_x*np.ones_like(X)
    V = current_y*np.ones_like(Y)

    water = ax.quiver(X, Y, U, V,
                      alpha=0.6)

    ax.legend()


    # UPDATE FUNCTION :-

    def update(frame):

        # Update trajectory
        traj_line.set_data(
            traj_x[:frame],
            traj_y[:frame]
        )

        # Update boat
        new_boat = create_boat(
            traj_x[frame],
            traj_y[frame],
            headings[frame]
        )

        boat_patch.set_xy(new_boat)

        return traj_line, boat_patch

    # CREATE ANIMATION :-

    anim = FuncAnimation(
        fig,
        update,
        frames=len(t),
        interval=60,
        blit=False
    )

    plt.close(fig)

    return HTML(anim.to_html5_video())

# Note: Change the value below based on the value you set on the slider 👇

In [15]:
# Change the numerical values here of kp, kd, current_x and current_y
animate_boat(
    kp=1.5,
    kd=1.5,
    current_x=0.1,
    current_y=0.3
)